# Thí nghiệm <model> / <method> / <expNNN>

Notebook này chạy một thí nghiệm rồi ghi kết quả. **Bấm Run all** là đủ.

Nó tự làm phần khó TRƯỚC khi chạy: kéo ĐÚNG bản code đã ghim ở cell đầu (nên chạy trên Colab hay trên máy cá nhân đều ra cùng kết quả), rồi kiểm dữ liệu, thiết bị và quyền ghi. Có gì chưa đúng thì nó **dừng và in ra danh sách việc phải sửa**, thay vì chạy nửa chừng rồi hỏng.

Cell đầu do `python scripts/pin.py` ghi - sửa tay sẽ bị ghi đè ở lần ghim sau. Các cell còn lại là bản mẫu trong `templates/experiment/notebook.ipynb`.

In [ ]:
# --- BẢN CODE ĐÃ GHIM (do scripts/pin.py ghi; sửa tay sẽ bị ghi đè) ---
REPO_URL = 'https://github.com/TrieuKhac-dev/SentimentX'
REPO_BRANCH = 'experiment'
REPO_SHA = 'fe180947f3506c259a2523e2887e8fab94b612c4'
EXP_DIR = 'qwen3-4b-instruct-2507/prompt-cot/exp001'


In [ ]:
# Chuẩn bị môi trường: kéo ĐÚNG bản code đã ghim rồi mới `import src`.
import pathlib
import sys

IN_COLAB = "google.colab" in sys.modules


def repo_root():
    """Gốc repo: nơi có `src/paths.py`.

    Trên Colab là thư mục code vừa kéo về; trên máy cá nhân thì tìm từ thư mục đang đứng đi lên,
    để notebook chạy được dù Jupyter mở ở đâu.
    """
    if IN_COLAB:
        return pathlib.Path("/content/SentimentX")
    here = pathlib.Path.cwd()
    for candidate in [here] + list(here.parents):
        if (candidate / "src" / "paths.py").is_file():
            return candidate
    return here


CODE_DIR = repo_root()
if str(CODE_DIR) not in sys.path:
    sys.path.insert(0, str(CODE_DIR))

from src import paths, repo, runtime

# Nạp biến môi trường TRƯỚC khi dùng: token DagsHub và gốc dữ liệu/kết quả nằm ở file `.env`
# (máy cá nhân) hoặc Colab Secrets + `<Drive>/env/.env.colab`. Không nạp thì token có trong máy mà
# phần ghi nhận vẫn báo "thiếu token" - một lỗi im lặng rất khó đoán.
drive = runtime.drive_dir() if IN_COLAB else None
env = runtime.load_env(colab_env_file=(runtime.drive_env_file() if drive else None))
print("Nơi chạy    :", runtime.env_name())
if IN_COLAB:
    print("Drive       :", drive or "CHƯA thấy (mount Drive rồi chạy lại ô này)")
print("Gốc dữ liệu :", paths.data_root())
print("Gốc kết quả :", paths.results_root())
print("Biến bắt buộc: có {} | thiếu {}".format(
    ", ".join(env["found"]) or "không có", ", ".join(env["missing"]) or "không thiếu"))

# Kéo đúng commit đã ghim. Lệch sha là DỪNG, chứ không chạy trên bản code không rõ là bản nào.
code = repo.prepare(REPO_URL, REPO_SHA, branch=REPO_BRANCH, dest=CODE_DIR, require_branch=True)
print("Code        : {} | {} | {}".format(code["action"], code["dir"], REPO_SHA[:8]))
for warning in code["warnings"]:
    print("  cảnh báo:", warning)


In [ ]:
# Cấu hình ĐANG DÙNG: in ra để người đọc bảng điểm sau này biết nó thuộc cấu hình nào.
from src import dataset, experiments, model_config, versioning

parts = [part for part in EXP_DIR.replace("\\", "/").split("/") if part]
model_id, method, exp_id = parts
result = experiments.load(model_id, method, exp_id)
config = result["config"]
ds = dataset.load_config(config["data"]["dataset"])
version_id = versioning.compute_id(ds)

print("Thí nghiệm  : {}/{}/{}".format(model_id, method, exp_id))
print("Dataset     : {} {} -> {}".format(ds["name"], ds.get("version"), version_id))
print("Vai         : {}".format(config["data"]["roles"]))
print("Prompt      : {}".format(config["prompt"]))
print("Bài toán    : label_space={}, neutral_policy={}, not_mentioned={}".format(
    config.get("label_space"), config.get("neutral_policy"), config.get("not_mentioned")))
print("Chấm điểm   : {} mẫu/bài (null = cả split), chỉ số {}".format(
    config.get("n"), config.get("scores")))
print("Ghi nhận    : tracker={}, experiment={}".format(
    config.get("tracker"), config.get("experiment")))
print("Model       : ngưỡng cắt {} token".format(model_config.max_length(model_id)[0]))
print("              nạp {}".format(model_config.inference(model_id)))
print("\nLớp đã hợp nhất (theo thứ tự):")
for label, path in result["layers"]:
    print("  {:<12} {}".format(label, path))
if result["overrides"]:
    print("Khoá bị lớp sau đè lên:")
    for row in result["overrides"]:
        print("  ", row[0])


In [ ]:
# KIỂM TRƯỚC khi nạp model. Một lượt val tốn hàng chục phút, nên mọi thứ phải đúng từ đầu.
from src import preflight

out_dir = paths.results_dir(model_id, method, exp_id, version_id)
report = preflight.run(result, ds=ds, version_id=version_id, model_id=model_id, out_dir=out_dir)
preflight.print_report(report)
print("\nThư mục kết quả:", out_dir)
print("Trạng thái     :", report["info"].get("mode"), "-", report["info"].get("mode_reason"))

if report["problems"]:
    raise SystemExit(
        "DỪNG: còn {} việc phải sửa (xem danh sách ở trên). Sửa xong thì chạy lại ô này.".format(
            len(report["problems"])))


In [ ]:
# CHẠY thí nghiệm rồi chấm điểm: gọi THƯ VIỆN của repo, không gọi script dòng lệnh và không chép
# lại logic. `src/experiment_run.py` là nơi DUY NHẤT biết cách chạy; cửa vào dòng lệnh (dùng khi
# muốn chạy nhanh ngoài notebook) cũng gọi đúng hai hàm dưới đây nên hai đường không thể lệch nhau.
from src import experiment_run, resume

plan = experiment_run.plan(result)      # KHÔNG cần GPU: thiếu file hay sai config là dừng ngay
print("Chế độ chạy:", plan["mode"], "-", plan["reason"])
if plan["mode"] == resume.MODE_STOP:
    raise SystemExit("DỪNG: " + plan["reason"])

# Chạy tiếp được: máy đứt giữa chừng thì chạy lại ô này, phần đã xong nằm trong
# `predictions/part_*.jsonl` và điểm số vẫn tính trên CẢ split.
run_result = experiment_run.run(plan)
print("\nChế độ:", run_result["mode"], "| thư mục:", run_result["out_dir"])


## Kết quả nằm ở đâu

Mỗi lần chạy một thư mục: `experiments/<model>/<method>/<expNNN>/results/<mã dữ liệu>/<hậu tố>`.

| File | Nội dung |
| --- | --- |
| `run.log` | từng bước đã chạy, kèm lí do khi dừng; tìm `[RUN] mode=RESUME` khi chạy tiếp |
| `run_meta.json` | bản ghi lần chạy: code, config, dữ liệu, các attempt |
| `metrics.json` | chỉ số, kèm cách chấm (`label_space`, `neutral_policy`, số ô neutral bị loại) |
| `metrics.csv` | bảng dài `aspect, sentiment, metric, value` để so với các lần chạy khác |
| `mispredictions.csv` | chỉ các ô đoán sai |
| `predictions/part_*.jsonl` | kết quả ghi dần; chạy lại thì đi tiếp từ đây |
| `errors.json` | CHỈ có khi lỗi: kiểu lỗi, vết gọi, thứ còn thiếu |

Trên DagsHub: mở địa chỉ **có đuôi `.mlflow`** (in ở ô cuối). Trang repo của DagsHub chỉ hiện dữ
liệu của DagsHub nên nhìn như rỗng - đó là hai trang khác nhau.

In [ ]:
# KẾT THÚC: nhắc lại chỗ cần xem, và việc cần làm trước khi giao.
from src import tracking, utils

print("Kết quả :", utils.rel(run_result["out_dir"]))
print("DagsHub :", tracking.base.dagshub_config().get("mlflow_uri"))
print("\nĐọc kết quả theo thứ tự: metrics.json (số chính) -> metrics.csv (so với lần chạy khác)")
print("-> run.log (đã chạy những gì) -> errors.json (nếu có).")
print("\nLưu ý: ghi lại file notebook này vào git (chỉ một file) rồi đẩy lên nhánh {}.\n"
      "Sau khi ghim thì KHÔNG sửa thư mục thí nghiệm nữa cho tới khi người nhận chạy xong."
      .format(REPO_BRANCH))
